# Day 13 — GroupBy + Merging & Joining in Pandas

> **Two datasets:** TechMart Sales (GroupBy) + Customers/Orders (Merging)

**Author:** Shaurab Kumar Jha  
**Date:** Day 13 of 90  
**Goal:** MNC-ready Python & Gen AI Engineer

## Table of Contents
1. [Setup & Datasets](#setup)
2. [GroupBy Basics — Single & Multiple Keys](#groupby-basics)
3. [GroupBy Attributes — len, size, groups, get_group, first/last/nth](#groupby-attrs)
4. [agg() — Multiple Aggregations](#agg)
5. [transform() — Group-wise Operations on Original Index](#transform)
6. [apply() on Groups — Custom Functions](#apply)
7. [Looping over Groups](#loop)
8. [pd.concat — Vertical & Horizontal](#concat)
9. [pd.merge — INNER JOIN](#inner)
10. [pd.merge — LEFT JOIN](#left)
11. [pd.merge — RIGHT JOIN](#right)
12. [pd.merge — OUTER JOIN](#outer)
13. [Self Join](#self)
14. [left_on / right_on + suffixes](#lefton)
15. [Chained Merge — 3 Tables](#chain)
16. [np.intersect1d / np.setdiff1d](#numpy)
17. [Full Business Case — Join + GroupBy](#business)
18. [Cheatsheet & Interview Q&A](#cheatsheet)

---
## 1. Setup & Datasets <a name='setup'></a>

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

Pandas: 2.2.3 | NumPy: 2.3.3


In [2]:
# ── TechMart Sales Dataset (for GroupBy) ────────────────────
n = 200
categories = ['Electronics', 'Clothing', 'Groceries', 'Books', 'Sports']
products   = {
    'Electronics': ['Laptop', 'Phone', 'Tablet', 'Headphones'],
    'Clothing':    ['Jacket', 'Shoes', 'T-Shirt', 'Jeans'],
    'Groceries':   ['Rice', 'Oil', 'Sugar', 'Pasta'],
    'Books':       ['Fiction', 'Science', 'History', 'Comics'],
    'Sports':      ['Bat', 'Ball', 'Gloves', 'Racket'],
}
price_map = {'Electronics':(200,1500),'Clothing':(20,300),
             'Groceries':(5,50),'Books':(10,60),'Sports':(15,200)}

category_col    = np.random.choice(categories, n)
product_col     = [np.random.choice(products[c]) for c in category_col]
units_col       = np.random.randint(1, 20, n)
price_col       = np.array([round(np.random.uniform(*price_map[c]),2) for c in category_col])
revenue_col     = (units_col * price_col).round(2)
month_order     = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

sales = pd.DataFrame({
    'Region':      np.random.choice(['North','South','East','West'], n),
    'Category':    category_col,
    'Product':     product_col,
    'Month':       np.random.choice(month_order, n),
    'Salesperson': np.random.choice(['Alice','Bob','Carol','Dave','Eve'], n),
    'Units':       units_col,
    'Price':       price_col,
    'Revenue':     revenue_col,
})

print(f'Sales dataset: {sales.shape}  |  Columns: {sales.columns.tolist()}')
sales.head(6)

Sales dataset: (200, 8)  |  Columns: ['Region', 'Category', 'Product', 'Month', 'Salesperson', 'Units', 'Price', 'Revenue']


,Region,Category,Product,Month,Salesperson,Units,Price,Revenue
0,West,Books,Science,Mar,Bob,2,35.67,71.34
1,South,Sports,Ball,Jun,Carol,10,58.15,581.50
2,East,Groceries,Oil,Jan,Bob,2,31.16,62.32
3,North,Sports,Gloves,Jul,Dave,17,174.68,2969.56
4,North,Sports,Gloves,Jan,Alice,8,177.87,1422.96
5,East,Clothing,Jeans,Sep,Dave,1,86.27,86.27


In [3]:
# ── Customers + Orders + Products (for Merging) ─────────────
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5, 6],
    'name':        ['Aisha', 'Rohan', 'Priya', 'Karan', 'Deepa', 'Arjun'],
    'city':        ['Delhi', 'Mumbai', 'Bangalore', 'Delhi', 'Chennai', 'Hyderabad'],
    'member_tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Silver', 'Bronze'],
})

orders = pd.DataFrame({
    'order_id':    [101, 102, 103, 104, 105, 106, 107, 108],
    'customer_id': [  1,   2,   1,   3,   2,   4,   7,   7],  # 7 = ghost customer!
    'product':     ['Laptop','Phone','Tablet','Headphones','Charger','Speaker','Keyboard','Mouse'],
    'amount':      [55000, 22000, 35000, 8000, 1500, 4500, 3200, 900],
    'status':      ['Delivered','Delivered','Shipped','Delivered','Cancelled','Delivered','Delivered','Shipped'],
})

products_df = pd.DataFrame({
    'product_name': ['Laptop','Phone','Tablet','Headphones','Speaker','Monitor'],
    'category':     ['Electronics','Electronics','Electronics','Accessories','Accessories','Electronics'],
    'price':        [55000, 22000, 35000, 8000, 4500, 18000],
    'stock':        [50, 200, 80, 150, 120, 30],
})

print('Customers:'); display(customers)
print('\nOrders:');   display(orders)

Customers:


,customer_id,name,city,member_tier
0,1,Aisha,Delhi,Gold
1,2,Rohan,Mumbai,Silver
2,3,Priya,Bangalore,Gold
3,4,Karan,Delhi,Bronze
4,5,Deepa,Chennai,Silver
5,6,Arjun,Hyderabad,Bronze



Orders:


,order_id,customer_id,product,amount,status
0,101,1,Laptop,55000,Delivered
1,102,2,Phone,22000,Delivered
2,103,1,Tablet,35000,Shipped
3,104,3,Headphones,8000,Delivered
4,105,2,Charger,1500,Cancelled
5,106,4,Speaker,4500,Delivered
6,107,7,Keyboard,3200,Delivered
7,108,7,Mouse,900,Shipped


---
## 2.GroupBy Basics <a name='groupby-basics'></a>

###  Theory — How GroupBy Works (Split-Apply-Combine)
```
df.groupby('Category')['Revenue'].sum()
        │                  │         │
      SPLIT            SELECT     COMBINE
  (group rows)       (column)  (aggregate)
```

**Step 1 — SPLIT:** Divide df into groups based on unique values of 'Category'
**Step 2 — APPLY:** Run an operation on each group (sum, mean, custom function)
**Step 3 — COMBINE:** Combine results back into a new DataFrame/Series

```python
# groupby returns a GroupBy OBJECT — not a DataFrame
# You must follow with .sum() / .mean() / .agg() etc.
grp = df.groupby('Category')          # GroupBy object
grp['Revenue'].sum()                  # NOW you get a result
```


In [4]:
# Single column groupby — total revenue per category
result = sales.groupby('Category')['Revenue'].sum().round(2)
print('Total Revenue per Category:')
print(result.sort_values(ascending=False))
print(f'\nType returned: {type(result)}')

Total Revenue per Category:
Category
Electronics    294019.70
Clothing        59739.96
Sports          48108.69
Books           17558.78
Groceries       10889.74
Name: Revenue, dtype: float64

Type returned: <class 'pandas.core.series.Series'>


In [5]:
# Multiple columns selected
print('Sum of Units AND Revenue per Category:')
sales.groupby('Category')[['Units', 'Revenue']].sum().sort_values('Revenue', ascending=False)

Sum of Units AND Revenue per Category:


,Units,Revenue
Category,,
Electronics,397,294019.70
Clothing,359,59739.96
Sports,389,48108.69
Books,517,17558.78
Groceries,346,10889.74


In [6]:
# Multiple groupby KEYS — 2 columns as group keys
# Creates a MultiIndex result
multi = sales.groupby(['Region', 'Category'])['Revenue'].sum().round(2)
print('Type:', type(multi))
print('Index type (MultiIndex):', type(multi.index))
print()
multi.head(10)

Type: <class 'pandas.core.series.Series'>
Index type (MultiIndex): <class 'pandas.core.indexes.multi.MultiIndex'>



Region  Category   
East    Books           4444.80
        Clothing        9283.68
        Electronics    42489.87
        Groceries       2246.59
        Sports          7711.07
North   Books           2893.82
        Clothing       32524.09
        Electronics    79999.41
        Groceries       1182.10
        Sports         18963.01
Name: Revenue, dtype: float64

In [7]:
# unstack() — converts inner index level to columns (pivot table style)
pivot = multi.unstack(fill_value=0).round(0)
print('Region × Category Revenue Matrix:')
pivot

Region × Category Revenue Matrix:


Category,Books,Clothing,Electronics,Groceries,Sports
Region,,,,,
East,4445.0,9284.0,42490.0,2247.0,7711.0
North,2894.0,32524.0,79999.0,1182.0,18963.0
South,6025.0,10573.0,87934.0,2873.0,11865.0
West,4195.0,7359.0,83597.0,4588.0,9570.0


---
## 3.  GroupBy Attributes <a name='groupby-attrs'></a>

###  Key Attributes & Methods on GroupBy Object

| Method/Attr | What it does |
|-------------|-------------|
| `len(grp)` | Number of groups |
| `grp.size()` | Row count per group (includes NaN) |
| `grp.groups` | Dict: `{group_label: [row_indices]}` |
| `grp.get_group(key)` | Extract one group as DataFrame |
| `grp.first()` | First row of each group |
| `grp.last()` | Last row of each group |
| `grp.nth(n)` | nth row of each group (0-indexed) |
| `grp.describe()` | Summary stats per group |
| `grp.sample(n)` | n random rows per group |
| `grp.nunique()` | Count unique values per group |

**`size()` vs `count()` difference:**
- `size()` → total rows per group (includes NaN)
- `count()` → non-NaN values per group per column

In [8]:
grp = sales.groupby('Category')

print(f'len(grp) → number of groups: {len(grp)}')
print()
print('grp.size() — rows per group:')
print(grp.size())

len(grp) → number of groups: 5

grp.size() — rows per group:
Category
Books          49
Clothing       35
Electronics    43
Groceries      37
Sports         36
dtype: int64


In [9]:
# grp.groups — dictionary mapping each group label to its row indices
print('grp.groups (first 3 indices per group):')
for name, idx in grp.groups.items():
    print(f"  '{name}': {len(idx)} rows | first 5 idx: {list(idx[:5])}")

grp.groups (first 3 indices per group):
  'Books': 49 rows | first 5 idx: [0, 10, 14, 16, 19]
  'Clothing': 35 rows | first 5 idx: [5, 13, 15, 20, 27]
  'Electronics': 43 rows | first 5 idx: [18, 23, 24, 33, 38]
  'Groceries': 37 rows | first 5 idx: [2, 6, 7, 8, 11]
  'Sports': 36 rows | first 5 idx: [1, 3, 4, 9, 12]


In [10]:
# get_group — extract entire group as DataFrame
electronics = grp.get_group('Electronics')
print(f'Electronics group: {electronics.shape}')
electronics[['Product', 'Units', 'Price', 'Revenue']].head(5)

Electronics group: (43, 8)


,Product,Units,Price,Revenue
18,Headphones,8,942.91,7543.28
23,Tablet,19,1085.35,20621.65
24,Headphones,15,508.76,7631.40
33,Phone,5,619.49,3097.45
38,Laptop,5,1104.03,5520.15


In [11]:
# first / last / nth
print('first() — first row of each group:')
display(grp[['Product', 'Revenue']].first())

print('\nnth(2) — 3rd row (0-indexed) of each group:')
grp[['Product', 'Revenue']].nth(2)

first() — first row of each group:


,Product,Revenue
Category,,
Books,Science,71.34
Clothing,Jeans,86.27
Electronics,Headphones,7543.28
Groceries,Oil,62.32
Sports,Ball,581.50



nth(2) — 3rd row (0-indexed) of each group:


,Product,Revenue
4,Gloves,1422.96
7,Rice,347.93
14,History,185.57
15,Jeans,566.60
24,Headphones,7631.40


In [12]:
# describe() — gives full stats summary per group
print('Revenue describe per Category:')
grp['Revenue'].describe().round(2)

Revenue describe per Category:


,count,mean,std,min,25%,50%,75%,max
Category,,,,,,,,
Books,49.0,358.34,238.31,20.36,150.75,321.36,528.20,948.16
Clothing,35.0,1706.86,1583.52,86.27,315.73,1165.32,2759.34,5191.92
Electronics,43.0,6837.67,5907.58,858.33,2166.66,4699.05,9918.00,22497.90
Groceries,37.0,294.32,199.19,6.95,114.24,262.24,412.65,686.47
Sports,36.0,1336.35,991.68,68.80,412.17,1020.80,2354.85,3315.51


In [13]:
# sample — 1 random row per group
print('sample(1) — one random row per group:')
grp.sample(1, random_state=42)[['Category', 'Product', 'Revenue']]

sample(1) — one random row per group:


,Category,Product,Revenue
50,Books,Comics,564.87
27,Clothing,Jeans,820.99
33,Electronics,Phone,3097.45
30,Groceries,Pasta,246.54
74,Sports,Ball,1755.60


In [14]:
# nunique — how many unique products in each category?
print('nunique — unique products per category:')
grp['Product'].nunique()

nunique — unique products per category:


Category
Books          4
Clothing       4
Electronics    4
Groceries      4
Sports         4
Name: Product, dtype: int64

---
## 4.agg() — Multiple Aggregations <a name='agg'></a>

###  Theory — Three Ways to Call agg()

```python
# Way 1: List — same functions applied to selected column
grp['Revenue'].agg(['sum', 'mean', 'max'])

# Way 2: Dict — different functions per column
grp.agg({'Revenue': ['sum', 'mean'], 'Units': 'max'})

# Way 3: Named agg (cleanest — pandas 0.25+)
grp.agg(
    Total = pd.NamedAgg(column='Revenue', aggfunc='sum'),
    Avg   = pd.NamedAgg(column='Revenue', aggfunc='mean'),
)

# Way 4: Custom lambda inside agg
grp['Revenue'].agg(lambda x: x.max() - x.min())
```


In [15]:
# Way 1: List of functions
print('agg list — sum, mean, max on Revenue:')
sales.groupby('Category')['Revenue'].agg(['sum', 'mean', 'max', 'count']).round(2)

agg list — sum, mean, max on Revenue:


,sum,mean,max,count
Category,,,,
Books,17558.78,358.34,948.16,49
Clothing,59739.96,1706.86,5191.92,35
Electronics,294019.70,6837.67,22497.90,43
Groceries,10889.74,294.32,686.47,37
Sports,48108.69,1336.35,3315.51,36


In [16]:
# Way 2: Dict — different function per column
result_dict = sales.groupby('Category').agg({
    'Revenue': ['sum', 'mean'],
    'Units':   ['sum', 'max'],
    'Price':   'mean',
})
# Flatten MultiIndex column headers
result_dict.columns = ['Total_Rev', 'Avg_Rev', 'Total_Units', 'Max_Units', 'Avg_Price']
result_dict.round(2)

,Total_Rev,Avg_Rev,Total_Units,Max_Units,Avg_Price
Category,,,,,
Books,17558.78,358.34,517,19,33.51
Clothing,59739.96,1706.86,359,19,155.73
Electronics,294019.70,6837.67,397,19,730.80
Groceries,10889.74,294.32,346,19,30.31
Sports,48108.69,1336.35,389,19,118.59


In [17]:
# Way 3: Named Aggregations — cleanest, most readable
result_named = sales.groupby('Region').agg(
    Total_Revenue   = pd.NamedAgg(column='Revenue', aggfunc='sum'),
    Avg_Revenue     = pd.NamedAgg(column='Revenue', aggfunc='mean'),
    Total_Units     = pd.NamedAgg(column='Units',   aggfunc='sum'),
    Unique_Products = pd.NamedAgg(column='Product', aggfunc='nunique'),
    Best_Sale       = pd.NamedAgg(column='Revenue', aggfunc='max'),
).round(2)

print('Named agg by Region:')
result_named.sort_values('Total_Revenue', ascending=False)

Named agg by Region:


,Total_Revenue,Avg_Revenue,Total_Units,Unique_Products,Best_Sale
Region,,,,,
North,135562.43,2711.25,553,18,18154.69
South,119270.24,2092.46,584,19,18792.65
West,109308.19,2277.25,502,19,22497.90
East,66176.01,1470.58,369,17,20621.65


In [18]:
# Way 4: Custom lambda in agg
# Revenue range = max - min per category
rev_range = sales.groupby('Category')['Revenue'].agg(
    lambda x: x.max() - x.min()
).round(2)
rev_range.name = 'Revenue_Range'
print('Revenue Range (max-min) per Category:')
print(rev_range.sort_values(ascending=False))

Revenue Range (max-min) per Category:
Category
Electronics    21639.57
Clothing        5105.65
Sports          3246.71
Books            927.80
Groceries        679.52
Name: Revenue_Range, dtype: float64


---
## 5. transform() — Group-wise Operations <a name='transform'></a>

###  Theory — transform() vs apply() vs agg()

| Method | Returns | Use when |
|--------|---------|----------|
| `agg()` | One row per group | Summary statistics |
| `apply()` | Flexible (scalar/Series/DataFrame) | Complex custom logic |
| `transform()` | **Same shape as original df** | Group-wise element operations |

**transform() is perfect for:**
- Adding a new column that shows group-level stats alongside original rows
- Ranking within groups
- Normalizing within groups
- Z-scoring within groups

```python
# agg → collapses rows: 200 rows → 5 rows
sales.groupby('Category')['Revenue'].agg('mean')    # 5 rows

# transform → keeps shape: 200 rows → 200 rows
sales.groupby('Category')['Revenue'].transform('mean') # 200 rows, each = group mean
```

In [19]:
df_t = sales.copy()

# Add group mean alongside each row
df_t['Category_Avg_Revenue'] = sales.groupby('Category')['Revenue'].transform('mean').round(2)
df_t['Above_Category_Avg']   = df_t['Revenue'] > df_t['Category_Avg_Revenue']

print('Each row shows its category average revenue:')
df_t[['Category', 'Product', 'Revenue', 'Category_Avg_Revenue', 'Above_Category_Avg']].head(8)

Each row shows its category average revenue:


,Category,Product,Revenue,Category_Avg_Revenue,Above_Category_Avg
0,Books,Science,71.34,358.34,False
1,Sports,Ball,581.50,1336.35,False
2,Groceries,Oil,62.32,294.32,False
3,Sports,Gloves,2969.56,1336.35,True
4,Sports,Gloves,1422.96,1336.35,True
5,Clothing,Jeans,86.27,1706.86,False
6,Groceries,Pasta,412.65,294.32,True
7,Groceries,Rice,347.93,294.32,True


In [20]:
# Ranking within group using transform
df_t['Rank_in_Category'] = (sales.groupby('Category')['Revenue']
                                  .transform(lambda x: x.rank(ascending=False, method='dense')))

# Min-Max normalization within group
df_t['Revenue_Normalized'] = (sales.groupby('Category')['Revenue']
                                    .transform(lambda x: ((x - x.min()) / (x.max() - x.min())).round(4)))

# Z-score within group
df_t['Revenue_Zscore'] = (sales.groupby('Category')['Revenue']
                               .transform(lambda x: ((x - x.mean()) / x.std()).round(4)))

print('Ranking + Normalization + Z-score within Category:')
df_t[['Category', 'Revenue', 'Rank_in_Category', 'Revenue_Normalized', 'Revenue_Zscore']]\
    .sort_values(['Category', 'Rank_in_Category']).head(10)

Ranking + Normalization + Z-score within Category:


,Category,Revenue,Rank_in_Category,Revenue_Normalized,Revenue_Zscore
79,Books,948.16,1.0,1.0000,2.4750
31,Books,883.50,2.0,0.9303,2.2037
192,Books,731.08,3.0,0.7660,1.5641
173,Books,710.24,4.0,0.7436,1.4766
68,Books,684.36,5.0,0.7157,1.3680
156,Books,679.44,6.0,0.7104,1.3474
19,Books,658.70,7.0,0.6880,1.2604
28,Books,619.14,8.0,0.6454,1.0944
132,Books,610.72,9.0,0.6363,1.0590
185,Books,576.00,10.0,0.5989,0.9133


---
## 6.apply() on Groups <a name='apply'></a>

###  Theory
- `apply()` passes each group (as a full DataFrame) to your function
- Your function can return a scalar, Series, or DataFrame
- More flexible than `agg()` / `transform()` but slower
- Use `group_keys=False` to avoid adding group key to the index


In [21]:
# apply returning a scalar — Revenue Coefficient of Variation per Category
def cv(group):
    """Coefficient of Variation = std/mean * 100 — measures consistency"""
    return round((group['Revenue'].std() / group['Revenue'].mean()) * 100, 2)

cv_result = sales.groupby('Category').apply(cv)
cv_result.name = 'Revenue_CV_%'
print('Revenue Coefficient of Variation (lower = more consistent):')
print(cv_result.sort_values())

Revenue Coefficient of Variation (lower = more consistent):
Category
Books          66.50
Groceries      67.68
Sports         74.21
Electronics    86.40
Clothing       92.77
Name: Revenue_CV_%, dtype: float64


In [22]:
# apply returning a Series — rich summary per group
def category_summary(group):
    return pd.Series({
        'Total_Revenue':    round(group['Revenue'].sum(), 2),
        'Avg_Revenue':      round(group['Revenue'].mean(), 2),
        'Best_Product':     group.loc[group['Revenue'].idxmax(), 'Product'],
        'Top_Salesperson':  group.groupby('Salesperson')['Revenue'].sum().idxmax(),
        'Num_Transactions': len(group),
    })

print('Rich summary per category using apply:')
sales.groupby('Category').apply(category_summary)

Rich summary per category using apply:


,Total_Revenue,Avg_Revenue,Best_Product,Top_Salesperson,Num_Transactions
Category,,,,,
Books,17558.78,358.34,History,Bob,49
Clothing,59739.96,1706.86,Jeans,Carol,35
Electronics,294019.70,6837.67,Headphones,Eve,43
Groceries,10889.74,294.32,Rice,Carol,37
Sports,48108.69,1336.35,Racket,Dave,36


---
## 7. Looping over Groups <a name='loop'></a>

In [23]:
# Iterate over groups: for name, group_df in grp
print('Regional Sales Report:')
print('-' * 65)
for name, group in sales.groupby('Region'):
    top = group.loc[group['Revenue'].idxmax()]
    print(f"  {name:6s} | Rows: {len(group):3d} | "
          f"Total: ₹{group['Revenue'].sum():>10,.0f} | "
          f"Best: {top['Product']} ₹{top['Revenue']:,.0f}")

Regional Sales Report:
-----------------------------------------------------------------
  East   | Rows:  45 | Total: ₹    66,176 | Best: Tablet ₹20,622
  North  | Rows:  50 | Total: ₹   135,562 | Best: Headphones ₹18,155
  South  | Rows:  57 | Total: ₹   119,270 | Best: Laptop ₹18,793
  West   | Rows:  48 | Total: ₹   109,308 | Best: Headphones ₹22,498


In [24]:
# Collect per-group DataFrames into a dict
group_dict = {name: grp_df for name, grp_df in sales.groupby('Category')}
print('Keys:', list(group_dict.keys()))
print('\nAccess Electronics group directly:')
group_dict['Electronics'].head(3)

Keys: ['Books', 'Clothing', 'Electronics', 'Groceries', 'Sports']

Access Electronics group directly:


,Region,Category,Product,Month,Salesperson,Units,Price,Revenue
18,North,Electronics,Headphones,Mar,Dave,8,942.91,7543.28
23,East,Electronics,Tablet,Apr,Dave,19,1085.35,20621.65
24,South,Electronics,Headphones,Jan,Eve,15,508.76,7631.40


---
## 8. pd.concat — Concatenation <a name='concat'></a>

###  Theory
```
pd.concat([df1, df2], axis=0)  → stack VERTICALLY   (add rows)
pd.concat([df1, df2], axis=1)  → stack HORIZONTALLY (add columns)

Key parameters:
  ignore_index=True  → reset index after concat
  keys=['a','b']     → label which source each row came from (MultiIndex)
  join='inner'       → only keep columns present in ALL DataFrames
  join='outer'       → keep all columns, NaN where missing (default)
```

In [25]:
# New batch of customers to add
new_customers = pd.DataFrame({
    'customer_id': [7, 8, 9],
    'name':        ['Vikram', 'Sneha', 'Raj'],
    'city':        ['Pune', 'Kolkata', 'Jaipur'],
    'member_tier': ['Gold', 'Silver', 'Gold'],
})

# ── Vertical concat WITHOUT ignore_index ─────────────────────
all_cust_raw = pd.concat([customers, new_customers], axis=0)
print('Without ignore_index — notice repeated index:')
display(all_cust_raw)
print(f'Index: {all_cust_raw.index.tolist()}')

Without ignore_index — notice repeated index:


,customer_id,name,city,member_tier
0,1,Aisha,Delhi,Gold
1,2,Rohan,Mumbai,Silver
2,3,Priya,Bangalore,Gold
3,4,Karan,Delhi,Bronze
4,5,Deepa,Chennai,Silver
5,6,Arjun,Hyderabad,Bronze
0,7,Vikram,Pune,Gold
1,8,Sneha,Kolkata,Silver
2,9,Raj,Jaipur,Gold


Index: [0, 1, 2, 3, 4, 5, 0, 1, 2]


In [26]:
# ── With ignore_index=True — clean continuous index ──────────
all_cust = pd.concat([customers, new_customers], axis=0, ignore_index=True)
print('With ignore_index=True — clean index:')
display(all_cust)
print(f'Index: {all_cust.index.tolist()}')

With ignore_index=True — clean index:


,customer_id,name,city,member_tier
0,1,Aisha,Delhi,Gold
1,2,Rohan,Mumbai,Silver
2,3,Priya,Bangalore,Gold
3,4,Karan,Delhi,Bronze
4,5,Deepa,Chennai,Silver
5,6,Arjun,Hyderabad,Bronze
6,7,Vikram,Pune,Gold
7,8,Sneha,Kolkata,Silver
8,9,Raj,Jaipur,Gold


Index: [0, 1, 2, 3, 4, 5, 6, 7, 8]


In [27]:
# ── keys — label each source ─────────────────────────────────
labeled = pd.concat([customers, new_customers], keys=['Existing', 'New'])
print('With keys — MultiIndex shows source:')
display(labeled)
print('\nAccess only existing customers:')
display(labeled.loc['Existing'])

With keys — MultiIndex shows source:


customer_id    name       city member_tier
Existing 0            1   Aisha      Delhi        Gold
         1            2   Rohan     Mumbai      Silver
         2            3   Priya  Bangalore        Gold
         3            4   Karan      Delhi      Bronze
         4            5   Deepa    Chennai      Silver
         5            6   Arjun  Hyderabad      Bronze
New      0            7  Vikram       Pune        Gold
         1            8   Sneha    Kolkata      Silver
         2            9     Raj     Jaipur        Gold


Access only existing customers:


,customer_id,name,city,member_tier
0,1,Aisha,Delhi,Gold
1,2,Rohan,Mumbai,Silver
2,3,Priya,Bangalore,Gold
3,4,Karan,Delhi,Bronze
4,5,Deepa,Chennai,Silver
5,6,Arjun,Hyderabad,Bronze


In [28]:
# ── Horizontal concat (axis=1) ────────────────────────────────
extra = pd.DataFrame({
    'loyalty_points': [1200, 450, 2300, 150, 800, 90],
    'preferred_pay':  ['UPI', 'Card', 'UPI', 'Cash', 'Card', 'UPI'],
})
cust_extended = pd.concat([customers, extra], axis=1)
print('Horizontal concat (axis=1):')
cust_extended

Horizontal concat (axis=1):


,customer_id,name,city,member_tier,loyalty_points,preferred_pay
0,1,Aisha,Delhi,Gold,1200,UPI
1,2,Rohan,Mumbai,Silver,450,Card
2,3,Priya,Bangalore,Gold,2300,UPI
3,4,Karan,Delhi,Bronze,150,Cash
4,5,Deepa,Chennai,Silver,800,Card
5,6,Arjun,Hyderabad,Bronze,90,UPI


In [29]:
# ── Mismatched columns — outer vs inner join ─────────────────
df_a = pd.DataFrame({'A': [1,2], 'B': [3,4]})
df_b = pd.DataFrame({'B': [5,6], 'C': [7,8]})

print('join=outer (default) — NaN fills gaps:')
display(pd.concat([df_a, df_b], ignore_index=True))

print('join=inner — only common columns:')
display(pd.concat([df_a, df_b], join='inner', ignore_index=True))

join=outer (default) — NaN fills gaps:


,A,B,C
0,1.0,3,NaN
1,2.0,4,NaN
2,NaN,5,7.0
3,NaN,6,8.0


join=inner — only common columns:


,B
0,3
1,4
2,5
3,6


---
## 9.INNER JOIN <a name='inner'></a>

###  Theory
```
INNER JOIN → Only rows where key exists in BOTH tables

customers:  1,2,3,4,5,6
orders:     1,2,3,4,7
INNER:      1,2,3,4       ← intersection only

Dropped:
  • Customers 5,6 (no orders)
  • Order from customer 7 (not in customers table)
```
SQL equivalent: `SELECT * FROM customers INNER JOIN orders ON customers.customer_id = orders.customer_id`

In [30]:
inner = pd.merge(customers, orders, on='customer_id', how='inner')
print(f'customers: {len(customers)} rows | orders: {len(orders)} rows | inner join: {len(inner)} rows')
print()
print('Result — only matched rows:')
display(inner[['customer_id', 'name', 'city', 'order_id', 'product', 'amount']])
print()
print('✅ Customers 5 (Deepa) & 6 (Arjun) EXCLUDED — no orders')
print('✅ Orders 107 & 108 (customer_id=7) EXCLUDED — no matching customer')

customers: 6 rows | orders: 8 rows | inner join: 6 rows

Result — only matched rows:


,customer_id,name,city,order_id,product,amount
0,1,Aisha,Delhi,101,Laptop,55000
1,1,Aisha,Delhi,103,Tablet,35000
2,2,Rohan,Mumbai,102,Phone,22000
3,2,Rohan,Mumbai,105,Charger,1500
4,3,Priya,Bangalore,104,Headphones,8000
5,4,Karan,Delhi,106,Speaker,4500



✅ Customers 5 (Deepa) & 6 (Arjun) EXCLUDED — no orders
✅ Orders 107 & 108 (customer_id=7) EXCLUDED — no matching customer


---
## 10.LEFT JOIN <a name='left'></a>

###  Theory
```
LEFT JOIN → ALL rows from LEFT table + matching from RIGHT
           No match on right → NaN

customers (LEFT):  1,2,3,4,5,6   ← ALL kept
orders (RIGHT):    1,2,3,4,7
LEFT JOIN:         1,2,3,4,5,6   ← 5 & 6 appear with NaN order columns

Most common join — "get all customers and their orders if any"
```

In [31]:
left = pd.merge(customers, orders, on='customer_id', how='left')
print(f'Left join: {len(left)} rows')
display(left[['customer_id', 'name', 'city', 'order_id', 'product', 'amount']])
print()
print('✅ Deepa & Arjun appear — with NaN in order columns (no orders)')
print('✅ customer_id=7 orders NOT included (7 not in left/customers table)')

Left join: 8 rows


,customer_id,name,city,order_id,product,amount
0,1,Aisha,Delhi,101.0,Laptop,55000.0
1,1,Aisha,Delhi,103.0,Tablet,35000.0
2,2,Rohan,Mumbai,102.0,Phone,22000.0
3,2,Rohan,Mumbai,105.0,Charger,1500.0
4,3,Priya,Bangalore,104.0,Headphones,8000.0
5,4,Karan,Delhi,106.0,Speaker,4500.0
6,5,Deepa,Chennai,NaN,NaN,NaN
7,6,Arjun,Hyderabad,NaN,NaN,NaN



✅ Deepa & Arjun appear — with NaN in order columns (no orders)
✅ customer_id=7 orders NOT included (7 not in left/customers table)


---
## 11.RIGHT JOIN <a name='right'></a>

###  Theory
```
RIGHT JOIN → ALL rows from RIGHT table + matching from LEFT
            No match on left → NaN

customers (LEFT):  1,2,3,4,5,6
orders (RIGHT):    1,2,3,4,7    ← ALL kept
RIGHT JOIN:        1,2,3,4,7   ← order from 7 appears with NaN customer cols

"Get all orders and customer info if available"
Note: RIGHT JOIN is rarely used — usually can be rewritten as LEFT JOIN
by swapping the table order.
```

In [32]:
right = pd.merge(customers, orders, on='customer_id', how='right')
print(f'Right join: {len(right)} rows')
display(right[['customer_id', 'name', 'city', 'order_id', 'product', 'amount']])
print()
print('✅ Orders 107 & 108 (customer_id=7) appear — with NaN in customer cols')
print('✅ Customers 5 & 6 (Deepa, Arjun) NOT included — not in orders table')

Right join: 8 rows


,customer_id,name,city,order_id,product,amount
0,1,Aisha,Delhi,101,Laptop,55000
1,2,Rohan,Mumbai,102,Phone,22000
2,1,Aisha,Delhi,103,Tablet,35000
3,3,Priya,Bangalore,104,Headphones,8000
4,2,Rohan,Mumbai,105,Charger,1500
5,4,Karan,Delhi,106,Speaker,4500
6,7,NaN,NaN,107,Keyboard,3200
7,7,NaN,NaN,108,Mouse,900



✅ Orders 107 & 108 (customer_id=7) appear — with NaN in customer cols
✅ Customers 5 & 6 (Deepa, Arjun) NOT included — not in orders table


---
## 12. OUTER JOIN (Full Outer) <a name='outer'></a>

###  Theory
```
OUTER JOIN → ALL rows from BOTH tables
             No match on either side → NaN

customers: 1,2,3,4,5,6
orders:    1,2,3,4,7
OUTER:     1,2,3,4,5,6,7   ← union of all keys

Use case: find EVERYTHING including unmatched rows from both sides
```

In [33]:
outer = pd.merge(customers, orders, on='customer_id', how='outer')
print(f'Outer join: {len(outer)} rows')
display(outer[['customer_id', 'name', 'city', 'order_id', 'product', 'amount']]
        .sort_values('customer_id'))
print()
print('✅ Deepa & Arjun appear (no orders) — NaN in order columns')
print('✅ Orders 107,108 (customer_id=7) appear — NaN in customer columns')

Outer join: 10 rows


,customer_id,name,city,order_id,product,amount
0,1,Aisha,Delhi,101.0,Laptop,55000.0
1,1,Aisha,Delhi,103.0,Tablet,35000.0
2,2,Rohan,Mumbai,102.0,Phone,22000.0
3,2,Rohan,Mumbai,105.0,Charger,1500.0
4,3,Priya,Bangalore,104.0,Headphones,8000.0
5,4,Karan,Delhi,106.0,Speaker,4500.0
6,5,Deepa,Chennai,NaN,NaN,NaN
7,6,Arjun,Hyderabad,NaN,NaN,NaN
8,7,NaN,NaN,107.0,Keyboard,3200.0
9,7,NaN,NaN,108.0,Mouse,900.0



✅ Deepa & Arjun appear (no orders) — NaN in order columns
✅ Orders 107,108 (customer_id=7) appear — NaN in customer columns


In [34]:
# Visual summary of all 4 joins
summary = pd.DataFrame({
    'Join Type':   ['INNER', 'LEFT', 'RIGHT', 'OUTER'],
    'how=':        ["'inner'", "'left'", "'right'", "'outer'"],
    'Rows':        [len(inner), len(left), len(right), len(outer)],
    'Keeps':       [
        'Only matched rows',
        'All left + matched right',
        'All right + matched left',
        'All rows from both tables',
    ],
    'SQL':         ['INNER JOIN', 'LEFT JOIN', 'RIGHT JOIN', 'FULL OUTER JOIN'],
})
summary

,Join Type,how=,Rows,Keeps,SQL
0,INNER,'inner',6,Only matched rows,INNER JOIN
1,LEFT,'left',8,All left + matched right,LEFT JOIN
2,RIGHT,'right',8,All right + matched left,RIGHT JOIN
3,OUTER,'outer',10,All rows from both tables,FULL OUTER JOIN


---
## 13.Self Join <a name='self'></a>

###  Theory
A **self join** merges a table with ITSELF.
Classic use case: employee → manager hierarchy (both in same table)

```
employees table:
  emp_id | emp_name | manager_id
     1   | Aisha    | NULL        ← CEO, no manager
     2   | Rohan    | 1           ← reports to Aisha
     3   | Priya    | 1           ← reports to Aisha

Self join: merge employees with employees[['emp_id','emp_name']]
           left_on='manager_id', right_on='emp_id'
           → adds manager_name column
```

In [35]:
employees = pd.DataFrame({
    'emp_id':     [1, 2, 3, 4, 5],
    'emp_name':   ['Aisha', 'Rohan', 'Priya', 'Karan', 'Deepa'],
    'manager_id': [np.nan, 1, 1, 2, 2],
    'dept':       ['Exec', 'Tech', 'Tech', 'Sales', 'Sales'],
})

display(employees)

# Self join to get manager names
hierarchy = pd.merge(
    employees,
    employees[['emp_id', 'emp_name']],     # right = same table (only id+name)
    left_on='manager_id',
    right_on='emp_id',
    how='left',
    suffixes=('', '_manager')             # avoid name collision
).rename(columns={'emp_name_manager': 'manager_name'})

print('\nHierarchy with manager names:')
hierarchy[['emp_id', 'emp_name', 'dept', 'manager_id', 'manager_name']]

,emp_id,emp_name,manager_id,dept
0,1,Aisha,NaN,Exec
1,2,Rohan,1.0,Tech
2,3,Priya,1.0,Tech
3,4,Karan,2.0,Sales
4,5,Deepa,2.0,Sales



Hierarchy with manager names:


,emp_id,emp_name,dept,manager_id,manager_name
0,1,Aisha,Exec,NaN,NaN
1,2,Rohan,Tech,1.0,Aisha
2,3,Priya,Tech,1.0,Aisha
3,4,Karan,Sales,2.0,Rohan
4,5,Deepa,Sales,2.0,Rohan


---
## 14.left_on / right_on + suffixes <a name='lefton'></a>

###  Theory
- `on='col'` → works when both tables have the SAME column name for the key
- `left_on='col_a', right_on='col_b'` → when key columns have DIFFERENT names
- `suffixes=('_x', '_y')` → when both tables have non-key columns with the same name


In [36]:
# left_on / right_on — different column names
orders_renamed = orders.rename(columns={'customer_id': 'cust_id'})
print('orders_renamed columns:', orders_renamed.columns.tolist())
print()

merged_diff = pd.merge(
    customers,
    orders_renamed,
    left_on='customer_id',    # column name in LEFT (customers)
    right_on='cust_id',       # column name in RIGHT (orders_renamed)
    how='inner'
)
print('Merged (both key cols appear — you can drop one):')
display(merged_diff[['customer_id', 'cust_id', 'name', 'product', 'amount']])

# Drop the duplicate key
merged_clean = merged_diff.drop(columns=['cust_id'])
print('\nAfter dropping cust_id:')
merged_clean[['customer_id', 'name', 'product', 'amount']]

orders_renamed columns: ['order_id', 'cust_id', 'product', 'amount', 'status']

Merged (both key cols appear — you can drop one):


,customer_id,cust_id,name,product,amount
0,1,1,Aisha,Laptop,55000
1,1,1,Aisha,Tablet,35000
2,2,2,Rohan,Phone,22000
3,2,2,Rohan,Charger,1500
4,3,3,Priya,Headphones,8000
5,4,4,Karan,Speaker,4500



After dropping cust_id:


,customer_id,name,product,amount
0,1,Aisha,Laptop,55000
1,1,Aisha,Tablet,35000
2,2,Rohan,Phone,22000
3,2,Rohan,Charger,1500
4,3,Priya,Headphones,8000
5,4,Karan,Speaker,4500


In [37]:
# suffixes — handle columns with same name in both tables
cust_with_status = customers.copy()
cust_with_status['status'] = ['Active','Active','VIP','Inactive','Active','New']
# Both cust_with_status and orders now have 'status'

merged_suffix = pd.merge(
    cust_with_status, orders,
    on='customer_id',
    how='inner',
    suffixes=('_customer', '_order')   # _customer for left, _order for right
)
print('With suffixes to resolve name clash:')
merged_suffix[['name', 'status_customer', 'status_order', 'amount']]

With suffixes to resolve name clash:


,name,status_customer,status_order,amount
0,Aisha,Active,Delivered,55000
1,Aisha,Active,Shipped,35000
2,Rohan,Active,Delivered,22000
3,Rohan,Active,Cancelled,1500
4,Priya,VIP,Delivered,8000
5,Karan,Inactive,Delivered,4500


---
## 15.Chained Merge — 3 Tables <a name='chain'></a>

In [38]:
# customers → orders → products (3-way join)
merged_3 = (
    pd.merge(customers, orders, on='customer_id', how='inner')       # step 1
      .merge(products_df,                                            # step 2
             left_on='product',
             right_on='product_name',
             how='left')
)

print(f'3-table join shape: {merged_3.shape}')
merged_3[['name', 'city', 'product', 'amount', 'category', 'stock']]

3-table join shape: (6, 12)


,name,city,product,amount,category,stock
0,Aisha,Delhi,Laptop,55000,Electronics,50.0
1,Aisha,Delhi,Tablet,35000,Electronics,80.0
2,Rohan,Mumbai,Phone,22000,Electronics,200.0
3,Rohan,Mumbai,Charger,1500,NaN,NaN
4,Priya,Bangalore,Headphones,8000,Accessories,150.0
5,Karan,Delhi,Speaker,4500,Accessories,120.0


---
## 16.np.intersect1d / np.setdiff1d <a name='numpy'></a>

###  Theory — Set Operations on Arrays

| Function | Returns | SQL equivalent |
|----------|---------|----------------|
| `np.intersect1d(a, b)` | Elements in BOTH a and b | INNER JOIN keys |
| `np.setdiff1d(a, b)` | Elements in a but NOT in b | LEFT ANTI JOIN |
| `np.union1d(a, b)` | All unique elements from both | OUTER JOIN keys |
| `np.in1d(a, b)` | Boolean mask: is each element of a in b? | `WHERE a IN b` |


In [39]:
cust_ids  = customers['customer_id'].values
order_ids = orders['customer_id'].unique()

print(f'Customer IDs: {cust_ids}')
print(f'Order CIDs:   {sorted(order_ids)}')
print()

Customer IDs: [1 2 3 4 5 6]
Order CIDs:   [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(7)]



In [40]:
# intersect1d — customers WHO HAVE placed orders
common = np.intersect1d(cust_ids, order_ids)
print(f'np.intersect1d → IDs in BOTH: {common}')
print('These customers have orders — equivalent to INNER JOIN keys')
print()

# setdiff1d — customers with NO orders
no_orders = np.setdiff1d(cust_ids, order_ids)
print(f'np.setdiff1d(customers, orders) → in customers NOT in orders: {no_orders}')
print('These customers have NEVER ordered')
print()

# setdiff1d other way — ghost orders
ghost = np.setdiff1d(order_ids, cust_ids)
print(f'np.setdiff1d(orders, customers) → in orders NOT in customers: {ghost}')
print('GHOST orders — customer doesnt exist in our system!')

np.intersect1d → IDs in BOTH: [1 2 3 4]
These customers have orders — equivalent to INNER JOIN keys

np.setdiff1d(customers, orders) → in customers NOT in orders: [5 6]
These customers have NEVER ordered

np.setdiff1d(orders, customers) → in orders NOT in customers: [7]
GHOST orders — customer doesnt exist in our system!


In [41]:
# union1d and in1d
all_ids = np.union1d(cust_ids, order_ids)
print(f'np.union1d → all unique IDs: {all_ids}')
print('OUTER JOIN would produce rows for all of these')
print()

# np.in1d — boolean mask (useful for filtering)
mask = np.in1d(cust_ids, order_ids)
print(f'np.in1d mask: {mask}')
print()
print('Customers who have ordered:')
customers[customers['customer_id'].isin(order_ids)]

np.union1d → all unique IDs: [1 2 3 4 5 6 7]
OUTER JOIN would produce rows for all of these

np.in1d mask: [ True  True  True  True False False]

Customers who have ordered:


,customer_id,name,city,member_tier
0,1,Aisha,Delhi,Gold
1,2,Rohan,Mumbai,Silver
2,3,Priya,Bangalore,Gold
3,4,Karan,Delhi,Bronze


---
## 17.Full Business Case <a name='business'></a>

In [42]:
# Q1: Revenue by city from DELIVERED orders only
delivered = (pd.merge(customers, orders, on='customer_id', how='inner')
               .query("status == 'Delivered'"))

city_revenue = (delivered.groupby('city')['amount']
                          .agg(Total_Revenue='sum', Num_Orders='count', Avg_Order='mean')
                          .round(2)
                          .sort_values('Total_Revenue', ascending=False))

print('Q1 — Revenue by City (Delivered orders only):')
display(city_revenue)

Q1 — Revenue by City (Delivered orders only):


,Total_Revenue,Num_Orders,Avg_Order
city,,,
Delhi,59500,2,29750.0
Mumbai,22000,1,22000.0
Bangalore,8000,1,8000.0


In [43]:
# Q2: Top spender per member tier
spenders = (pd.merge(customers, orders, on='customer_id', how='inner')
              .groupby(['member_tier', 'name'])['amount']
              .sum()
              .reset_index()
              .sort_values('amount', ascending=False)
              .groupby('member_tier')
              .first()   # top spender per tier
              .rename(columns={'name': 'Top_Customer', 'amount': 'Total_Spend'}))

print('Q2 — Top Customer per Member Tier:')
display(spenders)

Q2 — Top Customer per Member Tier:


,Top_Customer,Total_Spend
member_tier,,
Bronze,Karan,4500
Gold,Aisha,90000
Silver,Rohan,23500


In [44]:
# Q3: Sales leaderboard
leaderboard = sales.groupby('Salesperson').agg(
    Total_Revenue    = ('Revenue', 'sum'),
    Total_Units      = ('Units', 'sum'),
    Transactions     = ('Revenue', 'count'),
    Avg_Deal_Size    = ('Revenue', 'mean'),
    Regions_Covered  = ('Region', 'nunique'),
).round(2).sort_values('Total_Revenue', ascending=False)

print('Q3 — Salesperson Leaderboard:')
display(leaderboard)

Q3 — Salesperson Leaderboard:


,Total_Revenue,Total_Units,Transactions,Avg_Deal_Size,Regions_Covered
Salesperson,,,,,
Eve,98027.82,415,40,2450.70,4
Dave,91432.17,299,31,2949.42,4
Carol,85798.39,413,42,2042.82,4
Bob,78629.84,481,46,1709.34,4
Alice,76428.65,400,41,1864.11,4


In [45]:
# Q4: Monthly revenue bar chart (text-based)
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = sales.groupby('Month')['Revenue'].sum().reindex(month_order)

print('Q4 — Monthly Revenue:')
for m, rev in monthly.items():
    bar = '█' * int(rev // 600)
    print(f'  {m:3s}: ₹{rev:>9,.0f}  {bar}')

Q4 — Monthly Revenue:
  Jan: ₹   60,341  ████████████████████████████████████████████████████████████████████████████████████████████████████
  Feb: ₹   23,810  ███████████████████████████████████████
  Mar: ₹   26,191  ███████████████████████████████████████████
  Apr: ₹   47,700  ███████████████████████████████████████████████████████████████████████████████
  May: ₹   31,884  █████████████████████████████████████████████████████
  Jun: ₹   52,506  ███████████████████████████████████████████████████████████████████████████████████████
  Jul: ₹   32,616  ██████████████████████████████████████████████████████
  Aug: ₹   59,211  ██████████████████████████████████████████████████████████████████████████████████████████████████
  Sep: ₹   16,893  ████████████████████████████
  Oct: ₹   30,024  ██████████████████████████████████████████████████
  Nov: ₹   26,269  ███████████████████████████████████████████
  Dec: ₹   22,871  ██████████████████████████████████████


---
## 18.Cheatsheet & Interview Q&A <a name='cheatsheet'></a>

### GroupBy Cheatsheet
```python
# Basic
df.groupby('col')['target'].sum()           # single key
df.groupby(['a','b'])['target'].mean()      # multi key
df.groupby('col').size()                    # row count per group

# Attributes
len(grp)                                    # number of groups
grp.groups                                  # {label: [indices]}
grp.get_group('X')                          # extract one group
grp.first() / grp.last() / grp.nth(n)      # specific rows
grp['col'].nunique()                        # unique per group

# agg
grp['col'].agg(['sum','mean'])              # list
grp.agg({'a':'sum', 'b':'mean'})            # dict
grp.agg(Alias=pd.NamedAgg('col','sum'))     # named

# transform (keeps original shape)
grp['col'].transform('mean')               # group mean per row
grp['col'].transform(lambda x: x.rank())  # rank within group

# apply (flexible)
grp.apply(custom_function)                 # pass whole group df
```

### Merge Cheatsheet
```python
pd.merge(left, right, on='key', how='inner')    # INNER JOIN
pd.merge(left, right, on='key', how='left')     # LEFT JOIN
pd.merge(left, right, on='key', how='right')    # RIGHT JOIN
pd.merge(left, right, on='key', how='outer')    # FULL OUTER JOIN

# Different column names
pd.merge(left, right, left_on='a', right_on='b')

# Resolve duplicate columns
pd.merge(left, right, on='key', suffixes=('_l','_r'))

# Alternative syntax
left.merge(right, on='key', how='inner')
```

### Interview Q&A

| Question | Answer |
|----------|--------|
| `size()` vs `count()` in groupby? | `size()` counts all rows including NaN; `count()` counts non-NaN per column |
| `agg()` vs `transform()` vs `apply()`? | `agg` → one row per group; `transform` → same shape as input; `apply` → flexible |
| When to use `transform`? | Adding group stats as a new column while keeping original row structure |
| INNER vs LEFT JOIN? | INNER = only matched; LEFT = all left rows, NaN where no right match |
| `left_on` / `right_on` vs `on`? | `on` = same name in both; `left_on/right_on` = different names |
| `np.intersect1d` vs `np.setdiff1d`? | intersect = in both; setdiff1d(a,b) = in a but NOT in b |
| What is a self join? | Merging a table with itself — used for hierarchical data (employee-manager) |
| `ignore_index=True` in concat? | Resets the index to 0,1,2... after concatenation |
| `keys` param in concat? | Creates a MultiIndex to track which source each row came from |
